# H5 · Ablations, AVeriTeC, calibration, bootstrap (Himasri)

Runs every H5 experiment that needs no new teammate file (zero-shot NLI for now):
- **Ablations** on MultiFC test + twins: `twin` (H4's λ) vs `notwin` (no twin loss), `twin_norel` (no reranker), `twin_noent` (no entity match); seeds 42/43/44.
- **E1 on AVeriTeC test** (second English test set): every model as `avt_<model>`.
- **E5 calibration**: fitted on MultiFC val, applied to MultiFC test and AVeriTeC test (`*_cal`).
- **Paired bootstrap** (1,000 resamples) for every comparison, plus the LaTeX/CSV tables.

When Rathi's R4 NLI features land, rerun step 4 with `--nli-dir features/nli_v1 --tag _r4`.

**Before you start:** Runtime → **T4 GPU** (only step 3 uses it); Colab secret `GH_TOKEN`. Step 4 takes ~30–45 min on CPU.

## 1. Setup

In [ ]:
!git clone -q https://github.com/himaallu/Group-Project.git /content/twincheck
%cd /content/twincheck
!git checkout -q himasri/H5-experiments
# Rathi's R2 + R3 files (until merged into main); NOT committed from here.
!git fetch -q origin rathi/R3-twins
!git checkout origin/rathi/R3-twins -- data/copies/en/train.jsonl data/copies/en/test.jsonl features/relevance_en.parquet features/entities_en.jsonl features/relevance_copies_en.parquet
!git reset -q data/copies features/relevance_en.parquet features/entities_en.jsonl features/relevance_copies_en.parquet
!pip install -q -r requirements.txt
from google.colab import userdata
!ls features/nli_zeroshot features/nli_zeroshot/copies_en

## 2. 10-row check: AVeriTeC NLI + prediction with the H4 twin model
Outputs go to `/content/test10`.

In [ ]:
!mkdir -p /content/test10 && head -10 data/processed/en/averitec_test.jsonl > /content/test10/avt10.jsonl
!python -m src.nli.score_zeroshot /content/test10/avt10.jsonl --out-dir /content/test10
!python -m src.verdict.predict --model models/verdict_twin_s42 --data /content/test10/avt10.jsonl --nli /content/test10/avt10.parquet --relevance features/relevance_en.parquet --out /content/test10/preds.jsonl
!head -3 /content/test10/preds.jsonl

## 3. Zero-shot NLI on AVeriTeC test (1,112 pairs, < 1 min)

In [ ]:
!python -m src.nli.score_zeroshot data/processed/en/averitec_test.jsonl --out-dir features/nli_zeroshot
!ls -la features/nli_zeroshot

## 4. All H5 runs (12 trainings + AVeriTeC + calibration + bootstrap)

In [ ]:
!python -u -m src.eval.run_h5 --no-wandb 2>&1 | grep -E "^===|written|done|early stop|Error|Traceback|File \""


## 5. Key comparisons (paste this into the chat with Claude)

In [ ]:
import json, pandas as pd
pd.set_option('display.width', 250)
summary = pd.read_csv('results/H5/summary.csv', index_col=0)
cols = [c for c in summary.columns if c.endswith('_mean')]
print((summary[cols] * 100).round(1).to_string())
boot = json.load(open('results/H5/bootstrap.json'))
print()
for pair, res in boot.items():
    print(pair)
    for metric, r in res.items():
        print(f"  {metric:24} diff {r['diff']*100:+6.1f}  [{r['ci_low']*100:+6.1f}, {r['ci_high']*100:+6.1f}]  p={r['p_value']:.3f}")

## 6. Report tables (all models in `results/`) → `reports/tables/`

In [ ]:
!python -m src.eval.report --results results --gold data/processed/en/multifc_test.jsonl data/copies/en/test.jsonl data/processed/en/averitec_test.jsonl --out reports/tables
!ls reports/tables

## 7. Push to `himasri/H5-experiments`
Fill in your GitHub email. Rathi's files are not pushed.

In [ ]:
GIT_NAME = 'Himasri'
GIT_EMAIL = 'YOUR_GITHUB_EMAIL'
token = userdata.get('GH_TOKEN')
!git config user.name "{GIT_NAME}"
!git config user.email "{GIT_EMAIL}"
!git add features/nli_zeroshot/averitec_test.parquet results/H5 models/H5 reports/tables
!git status --short | grep -v '^??' | head -20
!git commit -q -m "[H5] Ablations, AVeriTeC test, calibration, bootstrap (zero-shot NLI, seeds 42/43/44)"
!git push -q https://{token}@github.com/himaallu/Group-Project.git himasri/H5-experiments && echo pushed

When it says `pushed`: Runtime → Disconnect and delete runtime.